<h1>Factory Machine Status</h1>

<hr/>

<p>Name: <strong>Goh Kun Ming</strong><br/></p>
<p>School: <strong>Singapore Polytechnic, School of Computing</strong><br/></p>
<p>Diploma: <strong>Diploma in Applied AI &amp; Analytics</strong><br/></p>
<p>Module: <strong>AI &amp; Machine Learning (ST1511)</strong><br/></p>
<p>Assessment: <strong>CA1 Part A</strong><br/></p>
<p>Academic Period: <strong>AY24/25 Year 1 Semester 2</strong><br/></p>
<p>Lecturer: <strong>Adjunct Lecturer Tai Hock Lin (Andy)</strong><br/></p>

<hr/>

<h1>Notebook Objective</h1>

<hr/>

<p>The objective of this notebook is to conduct feature engineering and statistical feature review. I will create the engineered features used by the model, verify that redundant source columns are removed after engineering, and run p-value based tests to understand feature association with machine status.</p>

<p>This notebook is part of the broken down notebook workflow. The original CA1 notebook is still maintained as <code>00_original_ca1_submission.ipynb</code>, while this notebook keeps the same report-style explanation and interpretation format in a smaller, easier-to-review file.</p>

<p>Notebook: <strong>02 Feature Engineering And Statistics</strong></p>


<hr/>
<h1>1.&nbsp;&nbsp;&nbsp;&nbsp;Importing of Modules</h1>
<hr/>

<p>Within this section, I will import the functions required for loading data, engineering features, and running statistical tests. These functions come from the production package so that the notebook follows the same workflow as the tested codebase.</p>


<h2>1.1&nbsp;&nbsp;&nbsp;&nbsp;Importing Feature Engineering and Statistical Test Utilities</h2>

<p>The following cell imports the constants and helper functions needed for this notebook.</p>


In [ ]:
import pandas as pd

from fault_prediction.config import DEFAULT_DATA_PATH, SOURCE_COLUMNS_DROPPED_AFTER_ENGINEERING
from fault_prediction.data import load_factory_data
from fault_prediction.features import engineer_features
from fault_prediction.statistics import run_statistical_tests


<p><strong>Interpretation:</strong> The imports above provide the feature engineering function and the statistical testing function. This keeps feature behavior consistent across notebooks, tests, and CLI commands.</p>


<hr/>
<h1>2.&nbsp;&nbsp;&nbsp;&nbsp;Feature Engineering</h1>
<hr/>

<p>Feature engineering is an important stage in machine learning because it can create more meaningful inputs for the model. In this project, feature engineering is used to create sensor-derived features that better represent machine operating conditions.</p>


<h2>2.1&nbsp;&nbsp;&nbsp;&nbsp;Importing Raw Data</h2>

<p>The raw dataset will be loaded before applying the feature engineering process.</p>


In [ ]:
df = load_factory_data(DEFAULT_DATA_PATH)
df.head()


<p><strong>Interpretation:</strong> The raw dataset still contains identifier columns, raw sensor columns, product quality, and the target column. The model should not directly use every raw column without feature selection and transformation.</p>


<hr/>
<h2>2.2&nbsp;&nbsp;&nbsp;&nbsp;Creating Engineered Features</h2>

<p>The production workflow creates two engineered features:</p>
<ul>
<li><code>Temperature Gradient = Process T (C) - Ambient T (C)</code></li>
<li><code>Power Indicator = Torque (Nm) / Rotation Speed (rpm)</code></li>
</ul>

<p>These features are designed to represent thermal load and torque relative to rotation speed.</p>


In [ ]:
engineered = engineer_features(df)
engineered.head(10)


<p><strong>Interpretation:</strong> The engineered dataframe now contains model-ready feature columns. It keeps <code>Tool Wear (min)</code>, <code>Quality</code>, <code>Temperature Gradient</code>, and <code>Power Indicator</code>.</p>


<hr/>
<h2>2.3&nbsp;&nbsp;&nbsp;&nbsp;Verifying Removal of Redundant Source Features</h2>

<p>In the original notebook, engineered features were created but some source columns remained in the model input. The improved workflow fixes this by removing columns that were only needed to create the engineered features.</p>


In [ ]:
removed_source_features = pd.DataFrame(
    {
        'Source Feature': list(SOURCE_COLUMNS_DROPPED_AFTER_ENGINEERING),
        'Removed From Model Input': [
            column not in engineered.columns
            for column in SOURCE_COLUMNS_DROPPED_AFTER_ENGINEERING
        ],
    }
)

assert removed_source_features['Removed From Model Input'].all()
removed_source_features


<p><strong>Interpretation:</strong> The verification confirms that the source features used to create engineered variables are removed from the final model input. This makes the feature set cleaner and easier to explain.</p>


<hr/>
<h1>3.&nbsp;&nbsp;&nbsp;&nbsp;Statistical Testing</h1>
<hr/>

<p>In this section, I will run statistical tests to review whether features have measurable association with the target variable. These tests provide evidence for feature understanding, but they do not replace model validation.</p>


<h2>3.1&nbsp;&nbsp;&nbsp;&nbsp;Running P-Value Tests</h2>

<p>The workflow applies Welch t-tests and Mann-Whitney U tests for numerical features, and a chi-square test for the categorical <code>Quality</code> feature.</p>


In [ ]:
statistical_tests = run_statistical_tests(df)
statistical_tests.head(15)


<p><strong>Interpretation:</strong> The p-value output helps identify which features show statistically significant differences or associations between normal and abnormal machine status. A low p-value suggests that the feature may be useful for distinguishing between classes.</p>


<hr/>
<h2>3.2&nbsp;&nbsp;&nbsp;&nbsp;Viewing Significant Results</h2>

<p>The next cell focuses on the key columns that are most useful for interpretation: feature name, test type, p-value, significance flag, and interpretation.</p>


In [ ]:
statistical_tests[
    ['feature', 'test', 'p_value', 'significant', 'interpretation']
].sort_values('p_value').head(15)


<p><strong>Interpretation:</strong> Statistical significance should be treated as one layer of evidence. A statistically significant feature may still need careful model validation, and a non-significant feature may still contribute through interactions with other variables.</p>


<hr/>
<h1>4.&nbsp;&nbsp;&nbsp;&nbsp;Notebook Summary</h1>
<hr/>

<p>From this notebook, I can verify that feature engineering is performed consistently and that redundant source columns are removed after engineered features are created. The statistical tests provide additional evidence for understanding feature relationships with machine status. The next notebook will focus on model training and threshold selection.</p>
